# 09 · Streamlit LLM App

Bring a real LLM into the Streamlit app, with chat history.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/09_streamlit_llm_app.ipynb)

**Covered in this notebook**

- Bringing real LLM into the streamlit website

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

# Bringing real LLM into the streamlit website

In [ ]:
!pip install langchain langchain_groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00


In [ ]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 113.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq

st.set_page_config(page_title='My AI Chat', layout='centered')

st.title("🤖 The Groq Chatbot")
st.write("A fully integrated, memory-enabled AI assistant.")

# The Sidebar (API Key Security)
with st.sidebar:
    st.header("⚙️ Configuration")
    user_api_key = st.text_input('Enter the Groq API Key:', type='password')
    st.info('Your key is required to wake up the AI Brain')

# ------ 2. The Memory Vault -------
if 'messages' not in st.session_state:
    st.session_state.messages = []

# ----- 3. Display History ---------
for msg in st.session_state.messages:
    with st.chat_message(msg['role']): # This display human and ai message differently
        st.markdown(msg['content'])

# 4. Chat Input and Logic
if user_query := st.chat_input('Say something to the AI...'):
# := -> operator waits until the user enters the message and then stores it

    if not user_api_key:
        st.error('Please enter your api key in the sidebar first!')
    else:
        # Display the user message instantly
        with st.chat_message('user'):
            st.markdown(user_query)

        # Store the user messsages to the vault
        st.session_state.messages.append({'role':'user', 'content': user_query})


        # Initialise the Brain
        llm = ChatGroq(
            model = 'openai/gpt-oss-20b',
            temperature = 0.7, # Creativity level of the model
            api_key = user_api_key
        )

        with st.spinner('AI is thinking....'):
            response = llm.invoke(st.session_state.messages)
            bot_answer = response.content


        st.session_state.messages.append({'role':'assistant', 'content':bot_answer})
        with st.chat_message('assistant'):
            st.markdown(bot_answer)

Overwriting app.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://YOUR-DOMAIN.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
# Practice Challenge: "The System Prompt Injector"
# Scenario: You have a working chatbot, but it is generic.
#You want to give the user the ability to define the AI's "Persona"
#(e.g., "You are a pirate" or "You are a strict math teacher") using a sidebar text area.

# The Logic Puzzle: LangChain expects the System Prompt to be the very first message in the history.
#We need to insert a {"role": "system", "content": persona} into the vault, but we only want to do it once!

# Your Task:

# In the sidebar, add persona = st.text_area("System Prompt:", value="You are a helpful assistant.").

# Add a button in the sidebar called "Reset Chat & Apply Persona".

# If that button is clicked, clear the vault (st.session_state.messages = []) so the chat restarts with the new persona.

# Before llm.invoke() happens, ensure the first message sent to the LLM includes the persona.

In [ ]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq

st.set_page_config(page_title='My AI Chat', layout='centered')

st.title("🤖 The Groq Chatbot")
st.write("A fully integrated, memory-enabled AI assistant.")

# The Sidebar (API Key + Persona)
with st.sidebar:
    st.header("⚙️ Configuration")

    user_api_key = st.text_input('Enter the Groq API Key:', type='password')

    persona = st.text_area(
        "System Prompt:",
        value="You are a helpful assistant."
    )

    if st.button("Reset Chat & Apply Persona"):
        st.session_state.messages = []
        st.rerun()

    st.info('Your key is required to wake up the AI Brain')


# ------ 2. The Memory Vault -------
if 'messages' not in st.session_state:
    st.session_state.messages = []


# ----- 3. Display History ---------
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])


# 4. Chat Input and Logic
if user_query := st.chat_input('Say something to the AI...'):

    if not user_api_key:
        st.error('Please enter your api key in the sidebar first!')

    else:
        # Display the user message instantly
        with st.chat_message('user'):
            st.markdown(user_query)

        # Store the user message to the vault
        st.session_state.messages.append({
            'role': 'user',
            'content': user_query
        })

        # Initialise the Brain
        llm = ChatGroq(
            model='openai/gpt-oss-20b',
            temperature=0.7,
            api_key=user_api_key
        )

        # Add System Prompt as the first message
        messages_for_llm = [
            {
                'role': 'system',
                'content': persona
            }
        ] + st.session_state.messages

        with st.spinner('AI is thinking....'):
            response = llm.invoke(messages_for_llm)
            bot_answer = response.content

        # Store assistant response
        st.session_state.messages.append({
            'role': 'assistant',
            'content': bot_answer
        })

        # Display assistant response
        with st.chat_message('assistant'):
            st.markdown(bot_answer)

Overwriting app.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://YOUR-DOMAIN.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
%%writefile requirements.txt
streamlit
langchain-groq

Writing requirements.txt


In [ ]:
# 1. The IoT Wearable "Context" Engine
# The Scenario: Smartwatches collect massive amounts of data, but mostly just display it. They rarely take autonomous action based on deep context. The Challenge: Build an edge-simulated, multi-agent system that acts as the "brain" for a wearable device.
# The system ingests a continuous, mock JSON stream of telemetry (e.g., heart rate, accelerometer data, GPS location, active app states like a custom beads-counting mini-program, and battery level).
# The Profiler Agent: Constantly updates a rolling window of the user's current state (e.g., "User is sleeping," "User is highly stressed and commuting").
# The Action Agent: Must decide when to interrupt the user. If the battery is at 5% but the user is navigating an unfamiliar city, it stays quiet. If the user's heart rate spikes while sitting completely still, it triggers an emergency protocol.

In [ ]:
%%writefile app.py

import streamlit as st
from langchain_groq import ChatGroq
import json

st.title("⌚ IoT Wearable Context Engine")

api_key = st.text_input("Groq API Key", type="password")

heart_rate = st.number_input("Heart Rate", value=75)
movement = st.selectbox("Movement", ["Low", "Medium", "High"])
battery = st.slider("Battery", 0, 100, 50)
location = st.text_input("Location", "Mumbai")
app_state = st.selectbox(
    "Active App",
    ["None", "Navigation", "Music", "Fitness"]
)

if st.button("Analyze"):

    if not api_key:
        st.error("Enter API Key")

    else:

        llm = ChatGroq(
            model="openai/gpt-oss-20b",
            temperature=0,
            api_key=api_key
        )

        telemetry = {
            "heart_rate": heart_rate,
            "movement": movement,
            "battery": battery,
            "location": location,
            "app": app_state
        }

        # Profiler Agent
        profiler_prompt = f"""
You are a Profiler Agent for a smartwatch.

Analyze this telemetry:

{json.dumps(telemetry)}

Determine the user's current state.
Give a short answer.
"""

        profiler = llm.invoke(profiler_prompt)

        st.subheader("Profiler Agent")
        st.write(profiler.content)

        # Action Agent
        action_prompt = f"""
You are an Action Agent for a smartwatch.

Telemetry:
{json.dumps(telemetry)}

User State:
{profiler.content}

Rules:

1. If heart rate is very high and movement is low,
trigger emergency protocol.

2. If battery is 5% or lower and navigation is active,
do not interrupt the user.

3. Otherwise, do not interrupt unnecessarily.

Tell me the final action and reason.
"""

        action = llm.invoke(action_prompt)

        st.subheader("Action Agent")
        st.success(action.content)

Overwriting app.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://YOUR-DOMAIN.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
# 2. The Algorithmic Instructional Designer
# The Scenario: Most "AI Tutors" just act as standard chatbots that output walls of text. True pedagogy requires structure, pacing, and verification. The Challenge: Build an autonomous curriculum generation system that ingests raw, dry technical documentation (e.g., a new software library) and transforms it into a highly structured lesson.
# The Architect Agent: Must map the raw data to strict instructional design frameworks (like Gagne’s Nine Events of Instruction or Merrill's First Principles), creating an outline.
# The Content Agent: Generates the actual teaching material, practical exercises, and assessments based on the Architect's blueprint.
# The Simulated Student Agent: Actively tries to complete the generated exercises using a separate LLM context. If it fails the assessment, the Architect and Content agents must review the failure logs and dynamically rewrite the confusing parts of the lesson.


In [ ]:
%%writefile app.py

import streamlit as st
from langchain_groq import ChatGroq

st.set_page_config(
    page_title="AI Instructional Designer",
    layout="centered"
)

st.title("📚 Algorithmic Instructional Designer")

# API Key
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input("Groq API Key", type="password")

# Documentation
documentation = st.text_area(
    "📄 Enter Technical Documentation",
    height=250,
    placeholder="Paste technical documentation here..."
)

if st.button("🚀 Generate Lesson"):

    if not api_key:
        st.error("Please enter Groq API Key.")

    elif not documentation:
        st.warning("Please enter documentation.")

    else:

        llm = ChatGroq(
            model="openai/gpt-oss-20b",
            temperature=0.3,
            api_key=api_key
        )

        # 1. Architect Agent
        architect_prompt = f"""
You are an Instructional Architect.

Use Gagne's Nine Events of Instruction
to design a structured lesson.

Technical Documentation:

{documentation}

Create a lesson blueprint containing:

1. Learning objectives
2. Lesson introduction
3. Important concepts
4. Examples
5. Practical exercise
6. Assessment questions
7. Expected learning outcome

Keep the blueprint beginner-friendly.
"""

        architect_response = llm.invoke(architect_prompt)

        blueprint = architect_response.content

        st.subheader("🏗️ Architect Agent")
        st.write(blueprint)


        # 2. Content Agent
        content_prompt = f"""
You are a Content Agent.

Create a complete beginner-friendly lesson
using this Architect blueprint:

{blueprint}

Technical Documentation:

{documentation}

The lesson must contain:

- Simple explanations
- Examples
- Practical exercise
- Assessment questions

Do not make the explanation unnecessarily long.
"""

        content_response = llm.invoke(content_prompt)

        lesson = content_response.content

        st.subheader("📖 Content Agent")
        st.write(lesson)


        # 3. Simulated Student Agent
        student_prompt = f"""
You are a beginner student.

Read the following lesson:

{lesson}

Try to solve the exercises and assessment
without receiving help.

At the end return exactly:

RESULT: PASS

or

RESULT: FAIL

If you fail, explain:

1. Which question was difficult
2. Why you could not answer it
3. What part of the lesson was confusing
"""

        student_response = llm.invoke(student_prompt)

        student_result = student_response.content

        st.subheader("🎓 Simulated Student")
        st.write(student_result)


        # 4. Check Student Result
        if "RESULT: FAIL" in student_result.upper():

            st.warning("❌ Student failed. Rewriting lesson...")

            # 5. Rewrite Agent
            rewrite_prompt = f"""
You are an Instructional Designer.

The original lesson was:

{lesson}

The simulated student's failure log was:

{student_result}

Rewrite only the confusing parts of the lesson.

Make them:
- Simpler
- Clearer
- More practical
- Easier for a beginner

Then provide an improved exercise
and assessment.
"""

            rewrite_response = llm.invoke(rewrite_prompt)

            improved_lesson = rewrite_response.content

            st.subheader("🔄 Improved Lesson")
            st.write(improved_lesson)

        else:

            st.success("✅ Student passed the assessment!")

Overwriting app.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://YOUR-DOMAIN.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
# 3. The "Smart City" Dynamic Dispatch Grid
# The Scenario: In a natural disaster, emergency call centers are overwhelmed, and civic resources (ambulances, fire trucks) are severely constrained. The Challenge: Create a real-time triage and dispatch swarm.
# The system receives a high-velocity stream of mocked 911 transcripts (unstructured, panicked text, sometimes containing conflicting information).
# The Triage Agent: Extracts the location, severity, and required resources, filtering out duplicate calls for the same event.
# The Dispatch Agent: Manages a strict database of available resources. It must route vehicles based on priority, calculating travel time (via a mock graph/map).

In [ ]:
%%writefile app.py

import streamlit as st
from langchain_groq import ChatGroq
import json

st.set_page_config(
    page_title="Smart City Dispatch",
    layout="centered"
)

st.title("🚑 Smart City Dynamic Dispatch Grid")


# API Key
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input(
        "Groq API Key",
        type="password"
    )


# Mock 911 calls
calls = st.text_area(
    "🚨 Emergency Calls",
    height=250,
    value="""
There is a huge fire near Andheri Station.
People are trapped inside the building.

Fire at Andheri Station.
Many people need help.

Someone is injured near Bandra.
Please send an ambulance.
"""
)


# Resource Database
resources = {
    "Fire Truck 1": {
        "type": "Fire Truck",
        "status": "Available"
    },

    "Fire Truck 2": {
        "type": "Fire Truck",
        "status": "Busy"
    },

    "Ambulance 1": {
        "type": "Ambulance",
        "status": "Available"
    },

    "Ambulance 2": {
        "type": "Ambulance",
        "status": "Available"
    }
}


# Mock Travel Times
travel_times = {
    "Fire Truck 1": {
        "Andheri Station": 5,
        "Bandra": 12
    },

    "Fire Truck 2": {
        "Andheri Station": 10,
        "Bandra": 15
    },

    "Ambulance 1": {
        "Andheri Station": 8,
        "Bandra": 4
    },

    "Ambulance 2": {
        "Andheri Station": 14,
        "Bandra": 7
    }
}


if st.button("🚨 Analyze & Dispatch"):

    if not api_key:

        st.error("Please enter Groq API Key.")

    else:

        llm = ChatGroq(
            model="openai/gpt-oss-20b",
            temperature=0,
            api_key=api_key
        )


        # =========================
        # 1. TRIAGE AGENT
        # =========================

        triage_prompt = f"""
You are an Emergency Triage Agent.

Analyze these emergency calls:

{calls}

For every unique incident identify:

- Location
- Emergency type
- Severity: LOW, MEDIUM or HIGH
- Number of people affected
- Required resource

Important:
Multiple calls may describe the same incident.
Identify duplicate calls and combine them.

Return a clear structured summary.
"""

        triage_response = llm.invoke(
            triage_prompt
        )

        incident_data = triage_response.content


        st.subheader("🚨 Triage Agent")

        st.write(incident_data)


        # =========================
        # 2. DISPATCH AGENT
        # =========================

        dispatch_prompt = f"""
You are an Emergency Dispatch Agent.

Here are the incidents identified by
the Triage Agent:

{incident_data}


Available resources:

{json.dumps(resources, indent=2)}


Mock travel times:

{json.dumps(travel_times, indent=2)}


Dispatch rules:

1. HIGH priority incidents first.
2. Only use AVAILABLE resources.
3. Match resource type with emergency.
4. Prefer the resource with the shortest travel time.
5. Do not assign one resource to two incidents.
6. Explain why each resource was selected.

Create a dispatch plan.

For each incident provide:

Incident:
Resource:
Travel Time:
Reason:
"""

        dispatch_response = llm.invoke(
            dispatch_prompt
        )


        st.subheader("🚒 Dispatch Agent")

        st.success(
            dispatch_response.content
        )

Overwriting app.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))

In [ ]:
import subprocess
import time

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give it a couple seconds to boot
time.sleep(3)

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8502)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://YOUR-DOMAIN.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
%%writefile requirements.txt
streamlit
langchain-groq

Overwriting requirements.txt


In [ ]:
from pyngrok import ngrok

ngrok.kill()